# Đánh giá baseline Dense bằng metric mới

Nạp checkpoint theo phase của các baseline Dense (`DenseIVNIDS`) và đánh giá bằng metric mới (F1 one-vs-rest từng lớp, FPR của normal), rồi đặt cạnh mô hình MoE chính để so sánh.

- Baseline được đánh giá lại từ checkpoint, nên dùng được cho cả các run huấn luyện bằng script cũ.
- Mô hình MoE tham chiếu được đọc thẳng từ `results.json` của run clean, vì run đó đã dùng metric mới.
- Có thể thêm baseline MoE fine-tune (arch `v2`) vào cùng danh sách.

Tập test mỗi phase được dựng giống hệt script huấn luyện. Cột `matches_log` kiểm tra overall F1 tính lại có khớp `results.json` của baseline không.

Notebook có thể đặt ở thư mục gốc hoặc trong `notebooks/`: thư mục gốc được tự tìm. Mọi đường dẫn trong cấu hình viết tính từ thư mục gốc. `dense_ivn_ids.py` nằm trong `networks/` hoặc thư mục gốc.

## 1. Cấu hình
Sửa đường dẫn cho khớp với nơi `train_baselines.py` đã lưu. `CKPT_PATTERN` là mẫu tên file checkpoint theo phase.

In [ ]:
PROJECT_ROOT = None        # None: tự tìm (thư mục chứa train_continual_clean.py)
DATA_DIR = "./data/2011-chevrolet-impala/preprocessed"
GPU = 0
BATCH_SIZE = 256
CKPT_PATTERN = "{phase}_best.pth"

BASELINES = [
    {"name": "dense_replay",   "arch": "dense", "dir": "./save/baselines/impala/dense_replay"},
    {"name": "dense_finetune", "arch": "dense", "dir": "./save/baselines/impala/dense_finetune"},
    # Baseline MoE fine-tune (không expand) nếu có:
    # {"name": "moe_finetune_replay", "arch": "v2", "dir": "./save/baselines/impala/moe_finetune_replay"},
]
# phase_config được đọc từ checkpoint; nếu checkpoint không lưu args, thêm "phase_config": "variant"

# Mô hình chính, đọc từ results.json (đã là metric mới)
REFERENCE = [
    {"name": "MoE_cosine_s42", "dir": "./save/clean/impala/cosine_s42", "phase_config": "variant"},
]

FOCUS_CLASSES = ["normal", "fuzzing", "rpm", "speed", "standstill", "combined"]

## 2. Import

In [ ]:
import os, sys, json, re
from pathlib import Path

# Tìm thư mục gốc project (nơi có train_continual_clean.py), bắt đầu từ thư mục notebook
# rồi đi ngược lên. Mọi đường dẫn "./data", "./save" trong cấu hình tính từ thư mục gốc.
NOTEBOOK_DIR = Path.cwd()
if "ROOT" not in globals():          # tránh tính lại khi chạy lại ô này
    if PROJECT_ROOT:
        ROOT = Path(PROJECT_ROOT).resolve()
    else:
        ROOT = next((p for p in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]
                     if (p / "train_continual_clean.py").exists()), None)
        if ROOT is None:
            raise FileNotFoundError("Không tìm thấy train_continual_clean.py ở thư mục notebook "
                                    "hoặc các thư mục cha; đặt PROJECT_ROOT thủ công")
    os.chdir(ROOT)
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
print("project root:", ROOT)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from train_continual_clean import (DEFAULT_PHASES, VARIANT_PHASES, filter_by_labels,
                                   split_dos_windows, per_class_f1)
from utils import (WindowDataset, collate_fn, compute_metrics, get_class_names,
                   load_metadata, load_windows)

DEVICE = torch.device(f"cuda:{GPU}" if torch.cuda.is_available() else "cpu")
CLASS_NAMES = get_class_names()
pd.set_option("display.precision", 4)
print("device:", DEVICE)

## 3. Dựng mô hình từ checkpoint

In [ ]:
from networks.moe_ivn_ids import MoEIVNIDSv2
try:
    from networks.moe_ivn_ids_cosine import MoEIVNIDS as MoEIVNIDSCosine
except ImportError:
    MoEIVNIDSCosine = None
try:
    from networks.dense_ivn_ids import DenseIVNIDS
except ImportError:
    try:
        from dense_ivn_ids import DenseIVNIDS
    except ImportError:
        DenseIVNIDS = None


def count_experts(sd):
    return len({k.split(".")[2] for k in sd if k.startswith("moe.experts.")})


def _num_classes(sd, default=10):
    idx = [int(m.group(1)) for k in sd if (m := re.match(r"classifier\.(\d+)\.weight$", k))]
    return sd[f"classifier.{max(idx)}.weight"].shape[0] if idx else default


def build_from_state(sd, arch, saved_args=None):
    """Dựng mô hình từ chính kích thước tensor trong checkpoint, rồi nạp strict.
    arch: "dense" | "v2" (cả v1 = v2 không bật cờ nào) | "cosine"."""
    saved_args = saved_args or {}
    vocab, embed_dim = sd["id_embedding.weight"].shape
    dropout = saved_args.get("dropout", 0.2)

    if arch == "dense":
        assert DenseIVNIDS is not None, "Không import được DenseIVNIDS"
        hidden, in_dim = sd["projection.0.weight"].shape
        model = DenseIVNIDS(vocab_size=vocab, num_features=in_dim - embed_dim,
                            embed_dim=embed_dim, num_classes=_num_classes(sd),
                            dense_hidden=hidden, out_dim=sd["projection.3.weight"].shape[0],
                            cls_hidden=sd["classifier.0.weight"].shape[0], dropout=dropout)
    else:
        expert0 = [k for k in sd if k.startswith("moe.experts.0.")]
        if any("spline" in k or "base_weight" in k for k in expert0):
            raise NotImplementedError("Checkpoint dùng KAN expert, notebook này chưa hỗ trợ")
        hidden, in_dim = sd["moe.experts.0.net.0.weight"].shape
        common = dict(vocab_size=vocab, num_features=in_dim - embed_dim, embed_dim=embed_dim,
                      num_experts=count_experts(sd), top_k=saved_args.get("top_k", 2),
                      moe_out_dim=sd["moe.experts.0.net.3.weight"].shape[0], dropout=dropout)
        if arch == "cosine":
            assert MoEIVNIDSCosine is not None, "Không import được moe_ivn_ids_cosine"
            model = MoEIVNIDSCosine(**common)
        elif arch in ("v1", "v2"):
            flags = dict(
                use_residual="residual_proj.weight" in sd,
                attention_readout=any(k.startswith("readout_attn.") for k in sd),
                light_classifier=not any(k.startswith("classifier.6.") for k in sd))
            model = MoEIVNIDSv2(**common, expert_hidden=hidden, num_classes=_num_classes(sd),
                                cls_hidden=sd["classifier.0.weight"].shape[0], **flags)
        else:
            raise ValueError(f"arch không hợp lệ: {arch}")
    model.load_state_dict(sd, strict=True)
    return model.eval()

In [ ]:
assert DenseIVNIDS is not None, "Không tìm thấy dense_ivn_ids.py trong networks/ hoặc thư mục gốc"

## 4. Tập test theo phase

In [ ]:
ALL_TEST = load_windows(Path(DATA_DIR) / "windows_test.pt")
_TEST_CACHE = {}

def phase_plan(phase_config):
    """Danh sách (tên phase, tập lớp đã thấy, test windows) giống hệt script huấn luyện."""
    key = phase_config
    if key in _TEST_CACHE:
        return _TEST_CACHE[key]
    phases = VARIANT_PHASES if phase_config == "variant" else DEFAULT_PHASES
    plan, seen = [], set()
    for i, ph in enumerate(phases):
        seen = seen | set(ph["attacks"])
        test = filter_by_labels(ALL_TEST, seen)
        if i == 0 and phase_config == "variant" and ph.get("use_dos_part") == 1:
            test = split_dos_windows(test, part=1)   # phase 1 chỉ có dos part 1
        loader = DataLoader(WindowDataset(test), BATCH_SIZE, shuffle=False,
                            collate_fn=collate_fn, num_workers=4)
        plan.append({"phase": ph["name"], "new": sorted(ph["attacks"]),
                     "type": ph.get("type", "new"), "seen": sorted(seen), "loader": loader})
    _TEST_CACHE[key] = plan
    return plan

## 5. Hàm đánh giá và tổng hợp

In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval()
    ys, ps = [], []
    for batch in loader:
        xl = [x.to(DEVICE) for x in batch["x_list"]]
        al = [a.to(DEVICE) for a in batch["adj_list"]]
        il = [i.to(DEVICE) for i in batch["id_idx_list"]]
        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            out = model(xl, al, il)
        ps.append(out["logits"].float().argmax(1).cpu())
        ys.append(batch["y"])
    return torch.cat(ys).numpy(), torch.cat(ps).numpy()


def new_metrics(y_true, y_pred, seen):
    """Metric mới: F1 one-vs-rest trên toàn bộ tập test + FPR của normal."""
    m = compute_metrics(y_true, y_pred)          # overall, giống log
    present = [c for c in seen if (y_true == c).any()]
    f1 = per_class_f1(y_true, y_pred, present)
    normal = y_true == 0
    return {
        "overall_f1": float(m["macro_f1"]), "acc": float(m["acc"]),
        "normal_fpr": float((y_pred[normal] != 0).mean()) if normal.any() else np.nan,
        **{CLASS_NAMES[c]: f1[c] for c in present},
    }


def load_state(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(ck, dict) and "model" in ck:
        return ck["model"], ck.get("args", {})
    return ck, {}


def load_results_json(run_dir):
    p = Path(run_dir) / "results.json"
    if not p.exists():
        return {}
    return {r["phase"]: r for r in json.load(open(p))}


def evaluate_run(run, ckpt_pattern="{phase}_best.pth"):
    """Đánh giá lại mọi checkpoint theo phase của một run bằng metric mới.
    Phase không có checkpoint (không huấn luyện) dùng lại mô hình của phase trước,
    nhưng vẫn được đánh giá trên tập test của chính phase đó."""
    run_dir = Path(run["dir"])
    _, saved_args = load_state(run_dir / ckpt_pattern.format(phase="phase1"))
    phase_config = run.get("phase_config") or saved_args.get("phase_config", "variant")
    old_json = load_results_json(run_dir)

    rows, sd = [], None
    for step in phase_plan(phase_config):
        path = run_dir / ckpt_pattern.format(phase=step["phase"])
        if path.exists():
            sd, _ = load_state(path)
            status = "loaded"
        elif sd is not None:
            status = "reused (no checkpoint)"
        else:
            raise FileNotFoundError(path)
        model = build_from_state(sd, run["arch"], saved_args).to(DEVICE)
        y, p = predict(model, step["loader"])
        del model

        row = {"run": run["name"], "phase": step["phase"], "type": step["type"],
               "status": status, "experts": count_experts(sd)}
        row.update(new_metrics(y, p, step["seen"]))
        logged = old_json.get(step["phase"], {}).get("overall", {}).get("macro_f1")
        row["log_overall_f1"] = logged
        row["matches_log"] = (abs(logged - row["overall_f1"]) < 5e-4) if logged is not None else None
        row["_old_per_class"] = old_json.get(step["phase"], {}).get("per_attack", {})
        row["_new_classes"] = [CLASS_NAMES[c] for c in step["new"]]
        rows.append(row)
        print(f"  [{run['name']} | {step['phase']}] {status:<22} F1={row['overall_f1']:.4f}"
              f"  log={logged if logged is not None else 'n/a'}")
    return pd.DataFrame(rows)

In [ ]:
def rows_from_results_json(ref):
    """Đọc run đã dùng metric mới, không cần nạp mô hình."""
    res = load_results_json(ref["dir"])
    rows = []
    for step in phase_plan(ref.get("phase_config", "variant")):
        r = res.get(step["phase"])
        if r is None:
            continue
        rows.append({"run": ref["name"], "phase": step["phase"], "type": step["type"],
                     "status": "results.json", "experts": r.get("num_experts"),
                     "overall_f1": r["overall"]["macro_f1"], "acc": r["overall"]["acc"],
                     "normal_fpr": r["overall"].get("normal_fpr", np.nan),
                     **r["per_attack"], "log_overall_f1": r["overall"]["macro_f1"],
                     "matches_log": True, "_old_per_class": {},
                     "_new_classes": [CLASS_NAMES[c] for c in step["new"]]})
    return pd.DataFrame(rows)

In [ ]:
def summarize(df):
    """Các chỉ số continual learning tính từ bảng per-class F1 (metric mới)."""
    out = []
    for run, g in df.groupby("run", sort=False):
        g = g.reset_index(drop=True)
        last = g.iloc[-1]
        learned, forgets, plastic = {}, [], []
        for i, r in g.iterrows():
            for c in r["_new_classes"]:
                if c not in learned and not np.isnan(r.get(c, np.nan)):
                    learned[c] = i
                    if i > 0 and r["type"] != "variant":
                        plastic.append(r[c])
        for c, i0 in learned.items():
            if i0 == len(g) - 1:
                continue
            hist = g.loc[i0:len(g) - 2, c].dropna()
            if len(hist):
                forgets.append((c, max(0.0, float(hist.max() - last[c]))))
        fmax = max(forgets, key=lambda t: t[1]) if forgets else (None, np.nan)
        out.append({
            "run": run,
            "final_overall_f1": last["overall_f1"],
            "final_mean_class_f1": float(np.nanmean([last[c] for c in learned])),
            "forgetting_max": fmax[1], "forgetting_max_class": fmax[0],
            "forgetting_mean": float(np.mean([f for _, f in forgets])) if forgets else np.nan,
            "new_class_f1_when_learned": float(np.mean(plastic)) if plastic else np.nan,
            "final_normal_fpr": last["normal_fpr"],
            "all_match_log": bool(g["matches_log"].dropna().all()) if g["matches_log"].notna().any() else None,
        })
    return pd.DataFrame(out).set_index("run")


def per_class_table(df, run):
    g = df[df["run"] == run].set_index("phase")
    cols = [c for c in CLASS_NAMES if c in g.columns and g[c].notna().any()]
    return g[cols]

In [ ]:
def plot_old_vs_new(df):
    """Metric cũ (results.json) so với metric mới, cho cùng checkpoint."""
    pts = []
    for _, r in df.iterrows():
        for c, old in r["_old_per_class"].items():
            if c in r and not pd.isna(r[c]):
                pts.append((r["run"], r["phase"], c, old, r[c]))
    if not pts:
        print("Không có results.json cũ để so sánh.")
        return
    p = pd.DataFrame(pts, columns=["run", "phase", "class", "old", "new"])
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    for run, g in p.groupby("run"):
        ax.scatter(g["old"], g["new"], s=22, alpha=0.75, label=run)
    lo = min(p["old"].min(), p["new"].min()) - 0.005
    ax.plot([lo, 1], [lo, 1], color="gray", ls="--", lw=1)
    ax.set_xlabel("F1 theo metric cũ (results.json)")
    ax.set_ylabel("F1 theo metric mới")
    ax.set_title("Mỗi điểm: một lớp tại một phase")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.show()
    p["inflation"] = p["old"] - p["new"]
    display(p.sort_values("inflation", ascending=False).head(10))


def plot_summary(summary):
    cols = ["final_overall_f1", "final_mean_class_f1", "new_class_f1_when_learned",
            "forgetting_max", "final_normal_fpr"]
    fig, axes = plt.subplots(1, len(cols), figsize=(4 * len(cols), 3.6), constrained_layout=True)
    for ax, c in zip(axes, cols):
        vals = summary[c]
        ax.barh(range(len(vals)), vals.values)
        ax.set_yticks(range(len(vals)), vals.index, fontsize=8)
        ax.set_title(c, fontsize=10)
        if "f1" in c:
            ax.set_xlim(max(0.0, np.nanmin(vals.values) - 0.02), 1.0)
        ax.grid(axis="x", alpha=0.3)
    plt.show()


def plot_class_trajectories(df, run):
    t = per_class_table(df, run)
    fig, ax = plt.subplots(figsize=(8, 4))
    for c in t.columns:
        ax.plot(range(len(t)), t[c], marker="o", label=c)
    ax.set_xticks(range(len(t)), t.index)
    ax.set_ylabel("F1 (metric mới)"); ax.set_title(run)
    ax.legend(fontsize=8, ncol=2, bbox_to_anchor=(1.01, 1), loc="upper left")
    ax.grid(alpha=0.3)
    plt.show()

## 6. Đánh giá baseline và nạp mô hình tham chiếu

In [ ]:
frames = []
for run in BASELINES:
    if not Path(run["dir"], CKPT_PATTERN.format(phase="phase1")).exists():
        print(f"[skip] {run['name']}: không thấy checkpoint phase1 trong {run['dir']}")
        continue
    try:
        d = evaluate_run(run, CKPT_PATTERN)
    except Exception as e:
        print(f"[FAIL] {run['name']}: {type(e).__name__}: {e}")
        continue
    d.drop(columns=[c for c in d.columns if c.startswith("_")]).to_csv(
        Path(run["dir"]) / "reeval_new_metric.csv", index=False)
    frames.append(d)

for ref in REFERENCE:
    if Path(ref["dir"], "results.json").exists():
        frames.append(rows_from_results_json(ref))
    else:
        print(f"[skip] {ref['name']}: không có results.json")

df = pd.concat(frames, ignore_index=True)

## 7. Kiểm tra tính đúng
Với các baseline, `matches_log` phải là True. Nếu baseline không có `results.json`, cột này để trống và cần đối chiếu tay với log.

In [ ]:
display(df[["run", "phase", "status", "overall_f1", "log_overall_f1", "matches_log"]])

## 8. F1 từng lớp theo phase

In [ ]:
for run in df["run"].unique():
    print(f"\n=== {run} ===")
    t = per_class_table(df, run)
    t = t.join(df[df["run"] == run].set_index("phase")[["overall_f1", "normal_fpr"]])
    display(t.style.format("{:.4f}", na_rep="").background_gradient(
        cmap="RdYlGn", vmin=0.9, vmax=1.0, subset=[c for c in t.columns if c != "normal_fpr"]))

## 9. Dense so với MoE

In [ ]:
summary = summarize(df)
display(summary)
summary.to_csv(NOTEBOOK_DIR / "baseline_summary.csv")
plot_summary(summary)

Diễn biến của các lớp trọng tâm, mỗi đường là một mô hình.

In [ ]:
classes = [c for c in FOCUS_CLASSES if c in df.columns]
ncol = 3
nrow = int(np.ceil(len(classes) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 3.3 * nrow),
                         constrained_layout=True, squeeze=False)
phases = sorted(df["phase"].unique(), key=lambda p: int(p.replace("phase", "")))
for ax, cls in zip(axes.flat, classes):
    for run, g in df.groupby("run", sort=False):
        g = g.set_index("phase").reindex(phases)
        ax.plot(range(len(phases)), g[cls], marker="o", label=run)
    ax.set_xticks(range(len(phases)), phases)
    ax.set_title(cls); ax.grid(alpha=0.3)
for ax in axes.flat[len(classes):]:
    ax.axis("off")
axes.flat[0].legend(fontsize=8)
plt.show()

In [ ]:
plot_old_vs_new(df[df['status'] != 'results.json'])

## 10. Đọc kết quả

- **Dense + replay xấp xỉ MoE** ở F1 cuối và forgetting: replay đang gánh phần lớn việc chống quên. Ưu thế của MoE cần được chứng minh qua thí nghiệm replay sensitivity (K nhỏ) hoặc chuỗi phase dài hơn.
- **MoE hơn rõ ở forgetting nhưng kém ở F1 lớp mới:** đây là đánh đổi stability-plasticity, cần báo cả hai chỉ số trong paper.
- **Dense fine-tune sụp:** khẳng định replay là bắt buộc, dùng làm cận dưới.
- Đặt `final_normal_fpr` cạnh F1 trong bảng chính: với IDS, báo động giả quan trọng không kém bỏ sót.